# Revision (R2/R3): training-only word selection
The word list is now chosen from the TRAINING sentences only (`MIN_EVAL_COUNT` is no longer used and the evaluation text is never read when choosing words), as in the v6 main run. The earlier version also required each word to occur in the evaluation text. The result tables replace the earlier ones. Output folder: `ablation_results_v2_trainonly`.


# Ablations of the calibrated ridge map (real vocabulary expansion, GPT-2 / OPT receivers)
Same pipeline, anchors, calibration data and scoring as the main real-vocabulary notebook. Variants:
A  no centring: ridge on raw rows (no intercept), direction = unit(prediction - receiver mean), calibrated scale; and the raw output as is.
B  magnitude: ridge output used with its own magnitude (no unit vector, no calibration); and the ridge direction with a fixed scale 1.0 (no calibration).
C  lambda: the ridge penalty chosen by the training NLL objective instead of held-out anchor cosine (all five values are run, so the sensitivity is visible).
Reference = ridge[all] (centred, unit direction, calibrated scale, lambda by held-out cosine). Needs internet (Kaggle: Internet On). Run top to bottom.

In [ ]:
%pip install -q transformers datasets scikit-learn

In [ ]:
import re, time, gc, random, pickle, traceback, logging
from typing import Tuple
from pathlib import Path
from collections import Counter
from types import SimpleNamespace
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from sklearn.linear_model import orthogonal_mp
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## Official OMP (mergekit batch_omp, verbatim)

In [ ]:
# Official batch_omp, copied verbatim from mergekit 0.1.5 (mergekit/tokensurgeon/omp.py). This is the function mergekit-tokensurgeon calls for '-a omp'.
# Copied (not imported) so that no extra package has to be installed on Kaggle. Only the function body is used.
LOG = logging.getLogger('batch_omp')

def batch_omp(
    targets: torch.Tensor,
    candidate_points: torch.Tensor,
    k: int,
    eps: float = 1e-8,
    reorthogonalize_interval: int = 256,
) -> Tuple[torch.LongTensor, torch.Tensor]:
    """
    Batched Orthogonal Matching Pursuit (OMP) to select `k` points from `candidate_points` that best approximate each target in `targets`.

    Args:
        targets: (B, D) tensor of target vectors.
        candidate_points: (N, D) tensor of candidate points.
        k: Number of points to select (sparsity level).
        eps: Tolerance for numerical stability.
        reorthogonalize_interval: Number of iterations between reorthogonalization steps.

    Returns:
        selected_indices: (B, k) tensor of indices selected for each target.
        coeff: (B, k) tensor of coefficients for each selected point.
    """
    B, D = targets.shape
    N, _ = candidate_points.shape
    device = targets.device
    if k > N:
        raise ValueError(f"Cannot select {k} points from {N} candidates")
    work_dtype = (
        targets.dtype
        if targets.dtype in (torch.float32, torch.float64)
        else torch.float32
    )
    # Convert inputs to work_dtype
    targets_work = targets.to(dtype=work_dtype)
    points_work = candidate_points.to(dtype=work_dtype)
    # Preallocate tensors
    q = torch.zeros((B, D, k), dtype=work_dtype, device=device)
    r = torch.zeros((B, k, k), dtype=work_dtype, device=device)
    selected_indices = torch.zeros((B, k), dtype=torch.long, device=device)
    mask = torch.zeros((B, N), dtype=torch.bool, device=device)
    residuals = targets_work.clone()

    for t in range(k):
        rms_0 = residuals.norm(dim=1).mean()
        # Compute absolute inner products between residuals and points
        abs_inner = (residuals @ points_work.T).abs()  # (B, N)
        # Mask out already selected points
        abs_inner.masked_fill_(mask, -float("inf"))

        # Select new index with maximum correlation
        _, new_idx = torch.max(abs_inner, dim=1)  # (B,)
        selected_indices[:, t] = new_idx
        mask[torch.arange(B, device=device), new_idx] = True

        new_atom = points_work[new_idx]  # (B, D)
        if t == 0:
            r[:, 0, 0] = new_atom.norm(dim=1)
            norm = r[:, 0, 0].clamp(min=eps)
            q[:, :, 0] = new_atom / norm.unsqueeze(1)
        else:
            # Project onto existing basis
            projections = torch.bmm(
                q[:, :, :t].transpose(1, 2), new_atom.unsqueeze(-1)
            ).squeeze(-1)  # (B, t)
            residual = new_atom - torch.bmm(
                q[:, :, :t], projections.unsqueeze(-1)
            ).squeeze(-1)  # (B, D)
            norm = torch.clamp(torch.norm(residual, dim=1), min=eps)
            # Update R and Q
            r[:, :t, t] = projections
            r[:, t, t] = norm
            q[:, :, t] = residual / norm.unsqueeze(-1)

        if t > 0 and t % reorthogonalize_interval == 0:
            q_b = q[:, :, : t + 1]
            q_new, r_new = torch.linalg.qr(q_b, mode="reduced")
            r[:, : t + 1, : t + 1] = torch.bmm(r_new, r[:, : t + 1, : t + 1])
            q[:, :, : t + 1] = q_new

        qt_targets = torch.bmm(
            q[:, :, : t + 1].transpose(1, 2), targets_work.unsqueeze(-1)
        )  # (B, t+1, 1)
        approx = torch.bmm(q[:, :, : t + 1], qt_targets).squeeze(-1)
        residuals = targets_work - approx
        LOG.debug(f"OMP iteration {t}: RMS {rms_0} -> {residuals.norm(dim=1).mean()}")

    # Get final coefficients
    final_coeff = torch.linalg.solve_triangular(
        r[:, :k, :k],
        torch.bmm(q[:, :, :k].transpose(1, 2), targets_work.unsqueeze(-1)),
        upper=True,
    ).squeeze(-1)

    # Print residuals if we're yapping
    if LOG.isEnabledFor(logging.DEBUG):
        rt_approx = torch.bmm(
            final_coeff.unsqueeze(1), points_work[selected_indices]
        ).squeeze(1)
        residuals = targets_work - rt_approx
        LOG.debug(f"OMP final RMS: {residuals.norm(dim=1).mean()}")

    return selected_indices, final_coeff


## Config

In [ ]:
SEEDS = [1, 2]             # both PubMed seeds run one after another in this session (finished runs are skipped on restart)
SEED = SEEDS[0]
DOMAIN = "pubmed"          # label used in the output file name ("pubmed" or "arxiv"); for arXiv also set DOMAIN_TXT to a file with one abstract per line
OMP64_K = 64                # the official tool's default
N_ABSTRACTS = 4000          # PubMedQA abstracts read (70% train / 30% eval, split by abstract)
TRAIN_FRAC = 0.7
K_TERMS = 300               # maximum number of new tokens added
MIN_TRAIN_COUNT, MIN_EVAL_COUNT = 5, 3     # a word must occur this often in train / eval text
MIN_TERMS = 50              # skip a pair with fewer eligible new words
MIN_PIECES = 2              # the receiver must split the word into at least this many tokens
OMP_K = 32
GRID = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5]    # embedding scale, relative to the mean receiver radius
N_TRAIN_T, N_TRAIN_C = 300, 150        # calibration sentences: with new words / without
N_EVAL_T, N_EVAL_C = 600, 200          # evaluation sentences: with new words / each control set
POOL_MAX = 2000                        # training pool for the optional new-row training
TRAIN_STEPS, TRAIN_LR, BATCH = 200, 1e-3, 8    # 0 = zero-shot only. Training changes ONLY the new rows
EVAL_BATCH = 16
USE_WIKI = True             # WikiText-2 as a general-text control (falls back to domain text only if it cannot load)
DOMAIN_TXT = None           # optional path: one abstract per line, to skip the PubMedQA download
OUT = Path("ablation_results_v2_trainonly"); OUT.mkdir(exist_ok=True)
PAIRS = [   # tied-embedding receivers only
    dict(name="GPT-2 <- Pythia-160m",     receiver="openai-community/gpt2", donor="EleutherAI/pythia-160m"),
    dict(name="GPT-2 <- Qwen2.5-0.5B",    receiver="openai-community/gpt2", donor="Qwen/Qwen2.5-0.5B"),
    dict(name="OPT-125m <- Pythia-160m",  receiver="facebook/opt-125m",     donor="EleutherAI/pythia-160m"),
    dict(name="OPT-125m <- Qwen2.5-0.5B", receiver="facebook/opt-125m",     donor="Qwen/Qwen2.5-0.5B"),
]

## Text, tokens, terms

In [ ]:
def split_sents(text):
    text = " ".join(text.split())
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if 8 <= len(s.split()) <= 60]

def load_domain():
    if DOMAIN_TXT:
        return [l.strip() for l in open(DOMAIN_TXT, encoding="utf-8") if l.strip()][:N_ABSTRACTS]
    from datasets import load_dataset
    for cfg, stream in (("pqa_artificial", True), ("pqa_labeled", False)):
        try:
            ds, out = load_dataset("qiaojin/PubMedQA", cfg, split="train", streaming=stream), []
            for ex in ds:
                out.append(" ".join(ex["context"]["contexts"]))
                if len(out) >= N_ABSTRACTS: break
            if out: print(f"domain text: PubMedQA/{cfg}, {len(out)} abstracts"); return out
        except Exception as e:
            print(f"PubMedQA/{cfg} failed: {type(e).__name__}: {str(e)[:150]}")
    raise RuntimeError("Could not load PubMedQA. Turn Internet on, or set DOMAIN_TXT to a text file with one abstract per line.")

def load_wiki():
    if not USE_WIKI: return []
    try:
        from datasets import load_dataset
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        text = " ".join(x for x in ds["text"][:30000] if x.strip() and not x.startswith(" ="))
        s = split_sents(text); print(f"general control: WikiText-2, {len(s)} sentences"); return s
    except Exception as e:
        print(f"WikiText-2 not available ({type(e).__name__}); only the domain control is used"); return []

SEG = re.compile(r" ?[^\W\d_]+| ?\d+| ?_+| ?[^\s\w]+|\s+")
def segments(s): return SEG.findall(s)

_ENC = {}
def enc(tok, seg):
    k = (id(tok), seg)
    if k not in _ENC: _ENC[k] = tok.encode(seg, add_special_tokens=False)
    return _ENC[k]

def build_seq(segs, tok, term_id, mode):
    ids, fl = [], []
    for g in segs:
        if mode == "new" and g in term_id: ids.append(term_id[g]); fl.append(1)
        else:
            p = enc(tok, g); ids += p; fl += [1 if g in term_id else 0] * len(p)
    return ids, fl

def mk(sents, tok, term_id, need_new):
    segs = [segments(s) for s in sents]
    o = [build_seq(g, tok, term_id, "orig") for g in segs]
    d = dict(sents=sents, o_ids=[x[0] for x in o], o_fl=[x[1] for x in o], n_ids=None, n_fl=None)
    if need_new:
        n = [build_seq(g, tok, term_id, "new") for g in segs]
        d["n_ids"], d["n_fl"] = [x[0] for x in n], [x[1] for x in n]
    return d

def pick_terms(rec_tok, don_tok, train_sents, eval_sents, V0r, V0d):
    ctr = Counter()
    for s in train_sents: ctr.update(g for g in segments(s)[1:] if re.fullmatch(r" [^\W\d_]{4,}", g))
    out = []
    for g, c in ctr.items():
        if c < MIN_TRAIN_COUNT: continue
        r, d = enc(rec_tok, g), enc(don_tok, g)
        if len(r) >= MIN_PIECES and len(d) == 1 and d[0] < V0d: out.append((c, g, d[0]))
    out.sort(key=lambda x: (-x[0], x[1]))
    return out[:K_TERMS]

## Models, scoring, maps

In [ ]:
def load_receiver(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name).float().to(DEVICE).eval()      # always float32: newer versions load the checkpoint's own precision (OPT is fp16)
    print(f"  receiver {name}: dtype {next(model.parameters()).dtype}")
    if model.get_output_embeddings().weight.data_ptr() != model.get_input_embeddings().weight.data_ptr():
        raise RuntimeError(f"{name}: input and output embeddings are not tied; this version supports tied receivers only")
    bos = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    return tok, model, bos

def load_donor(name):
    tok = AutoTokenizer.from_pretrained(name)
    m = AutoModel.from_pretrained(name)
    E = m.get_input_embeddings().weight.detach().float().cpu().numpy().copy()
    del m; gc.collect(); return tok, E

def make_batch(ids_list, fl_list, idx, bos):
    L = max(len(ids_list[i]) for i in idx) + 1
    x = torch.full((len(idx), L), bos, dtype=torch.long); m = torch.zeros((len(idx), L), dtype=torch.long); f = torch.zeros((len(idx), L - 1))
    for r, i in enumerate(idx):
        s = ids_list[i]; x[r, 1:1 + len(s)] = torch.tensor(s); m[r, :1 + len(s)] = 1; f[r, :len(s)] = torch.tensor(fl_list[i], dtype=torch.float)
    return x.to(DEVICE), m.to(DEVICE), f.to(DEVICE)

def batch_nll(model, x, m, limit=None):
    logits = model(input_ids=x, attention_mask=m).logits[:, :-1].float()
    if limit is not None: logits = logits[:, :, :limit]      # the unchanged receiver: ignore the added tokens completely
    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1), reduction="none").view(x.size(0), -1)
    return nll, m[:, 1:].float()

@torch.no_grad()
def score(model, bos, ids_list, fl_list, limit=None):
    n = len(ids_list); tot = np.zeros(n); term = np.zeros(n)
    order = sorted(range(n), key=lambda i: len(ids_list[i]))
    for b in range(0, n, EVAL_BATCH):
        idx = order[b:b + EVAL_BATCH]
        x, m, f = make_batch(ids_list, fl_list, idx, bos)
        nll, mm = batch_nll(model, x, m, limit)
        tot[idx] = (nll * mm).sum(1).cpu().numpy(); term[idx] = (nll * mm * f).sum(1).cpu().numpy()
    return tot, term

def set_rows(ctx, rows):
    w = ctx.model.get_input_embeddings().weight
    with torch.no_grad(): w[ctx.V0:ctx.V0 + ctx.K] = torch.as_tensor(np.asarray(rows, np.float32), dtype=w.dtype, device=w.device)

def unit(x, eps=1e-12): return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

def fit_ridge(Dc, Rc, seed=0):
    s = float(np.linalg.norm(Dc, axis=1).mean()); X = (Dc / s).astype(np.float64); Y = Rc.astype(np.float64)
    perm = np.random.default_rng(seed).permutation(len(X)); nv = min(max(50, len(X) // 10), len(X) // 5); va, tr = perm[:nv], perm[nv:]
    def solve(Xt, Yt, lam):
        A = Xt.T @ Xt; A[np.diag_indices_from(A)] += lam * np.trace(A) / A.shape[0]
        return np.linalg.solve(A, Xt.T @ Yt)
    best = None
    for lam in (1e-4, 1e-3, 1e-2, 1e-1, 1.0):
        cos = (unit(X[va] @ solve(X[tr], Y[tr], lam)) * unit(Y[va])).sum(1).mean()
        if best is None or cos > best[0]: best = (cos, lam)
    return solve(X, Y, best[1]), s, best[1]

def ridge_dirs(Dc, Rc, Dterm_c):
    W, s, lam = fit_ridge(Dc, Rc); return unit((Dterm_c / s) @ W), lam

def omp_dirs(Dc, Rc, Dterm_c, k):
    coef = orthogonal_mp(unit(Dc).astype(np.float64).T, unit(Dterm_c).astype(np.float64).T, n_nonzero_coefs=k)
    return unit(coef.reshape(len(Dc), -1).T @ unit(Rc))

def objective(ctx):
    t, c = ctx.D["t_train"], ctx.D["c_train"]
    tt, _ = score(ctx.model, ctx.bos, t["n_ids"], t["n_fl"]); cc, _ = score(ctx.model, ctx.bos, c["o_ids"], c["o_fl"])
    return 0.5 * tt.sum() / sum(map(len, t["o_ids"])) + 0.5 * cc.sum() / sum(map(len, c["o_ids"]))

def calibrate(ctx, direction):
    best = None
    for c in GRID:
        set_rows(ctx, ctx.mu_r + c * ctx.rbar * direction); o = objective(ctx)
        if best is None or o < best[1]: best = (c, o)
    return best

def evaluate(ctx, mode):
    r = {}
    for name in ("t_eval", "c_dom", "c_wiki"):
        S = ctx.D[name]
        if S is None: continue
        use_new = mode == "new" and S["n_ids"] is not None
        ids, fl = (S["n_ids"], S["n_fl"]) if use_new else (S["o_ids"], S["o_fl"])
        tot, term = score(ctx.model, ctx.bos, ids, fl, ctx.V0 if mode == "orig" else None); r[name] = dict(tot=tot, term=term, ntok=np.array(list(map(len, ids))))
    return r

def train_new_rows(ctx, rows, steps, seed):
    set_rows(ctx, rows)
    emb = ctx.model.get_input_embeddings().weight
    backup = emb.detach().clone()
    for p in ctx.model.parameters(): p.requires_grad_(False)
    emb.requires_grad_(True)
    opt = torch.optim.AdamW([emb], lr=TRAIN_LR, weight_decay=0.0); ctx.model.eval()
    P = ctx.D["t_pool"]; rng = random.Random(seed); skipped = 0
    for _ in range(steps):
        idx = [rng.randrange(len(P["n_ids"])) for _ in range(BATCH)]
        x, m, f = make_batch(P["n_ids"], P["n_fl"], idx, ctx.bos)
        nll, mm = batch_nll(ctx.model, x, m); loss = (nll * mm).sum() / mm.sum()
        if not torch.isfinite(loss): skipped += 1; opt.zero_grad(set_to_none=True); continue
        loss.backward(); emb.grad[:ctx.V0] = 0
        torch.nn.utils.clip_grad_norm_([emb], 1.0)
        if not bool(torch.isfinite(emb.grad).all()): skipped += 1; opt.zero_grad(set_to_none=True); continue
        opt.step(); opt.zero_grad(set_to_none=True)
    emb.requires_grad_(False)
    with torch.no_grad(): emb[:ctx.V0].copy_(backup[:ctx.V0])        # the original rows are restored exactly, whatever happened
    if skipped: print(f"      note: {skipped} of {steps} training steps skipped (non-finite loss or gradient)")

## One pair

In [ ]:
def run_pair(pair, seed=0):
    t_all = time.time(); name = pair["name"]
    rec_tok, model, bos = load_receiver(pair["receiver"]); don_tok, D_in = load_donor(pair["donor"])
    V0 = model.get_input_embeddings().weight.shape[0]; V0d = D_in.shape[0]
    E = model.get_input_embeddings().weight.detach().float().cpu().numpy().copy()
    abstracts = load_domain(); n_tr = int(len(abstracts) * TRAIN_FRAC)
    tr_s = [s for a in abstracts[:n_tr] for s in split_sents(a)]; ev_s = [s for a in abstracts[n_tr:] for s in split_sents(a)]
    terms = pick_terms(rec_tok, don_tok, tr_s, ev_s, V0, V0d)
    print(f"{name}: receiver dim {E.shape[1]}, donor dim {D_in.shape[1]} | eligible new words (donor 1 token, receiver >= {MIN_PIECES} pieces): {len(terms)}")
    if len(terms) < MIN_TERMS: print(f"  fewer than {MIN_TERMS} eligible words; skipped"); return None
    for _, g, _d in terms[:12]: print(f"    {g!r:<18} receiver pieces: {[rec_tok.convert_ids_to_tokens(i) for i in enc(rec_tok, g)]}")
    K = len(terms); term_id = {g: V0 + j for j, (_, g, _d) in enumerate(terms)}
    d_term = np.array([d for _, _g, d in terms]); piece_ids = [enc(rec_tok, g) for _, g, _d in terms]
    # anchors: whole words that are single tokens in both models (matched by token string)
    rv, dv = rec_tok.get_vocab(), don_tok.get_vocab()
    keys = sorted((k for k in rv if k in dv and k.startswith("Ġ") and k[1:].isalpha() and len(k) >= 3 and rv[k] < V0 and dv[k] < V0d), key=lambda k: rv[k])
    r_ids = np.array([rv[k] for k in keys]); d_ids = np.array([dv[k] for k in keys]); print(f"  shared anchor words: {len(keys)}")
    R_a, D_a = E[r_ids], D_in[d_ids]; mu_r, mu_d = R_a.mean(0), D_a.mean(0); Rc, Dc = R_a - mu_r, D_a - mu_d
    rbar = float(np.linalg.norm(Rc, axis=1).mean()); Dterm_c = D_in[d_term] - mu_d
    cnt = Counter(i for s in tr_s for g in segments(s) if g not in term_id for i in enc(rec_tok, g))
    f_idx = np.argsort([-cnt[i] for i in r_ids], kind="stable")[:min(5000, len(keys))]
    # sentence sets
    rng = random.Random(seed); has = lambda s: any(g in term_id for g in segments(s))
    tr_t = [s for s in tr_s if has(s)]; tr_c = [s for s in tr_s if not has(s)]
    ev_t = [s for s in ev_s if has(s)]; ev_c = [s for s in ev_s if not has(s)]
    ev_abs = {s: j for j, a in enumerate(abstracts[n_tr:]) for s in split_sents(a)}   # evaluation sentence -> its abstract (clusters for the bootstrap)
    wiki = [s for s in load_wiki() if not has(s)]
    smp = lambda L, n: rng.sample(L, min(n, len(L)))
    c_train_s = smp(tr_c, N_TRAIN_C // 2 if wiki else N_TRAIN_C) + (smp(wiki[:len(wiki) // 2], N_TRAIN_C // 2) if wiki else [])
    D = dict(t_train=mk(smp(tr_t, N_TRAIN_T), rec_tok, term_id, True), c_train=mk(c_train_s, rec_tok, term_id, False),
             t_pool=mk(smp(tr_t, POOL_MAX), rec_tok, term_id, True), t_eval=mk(smp(ev_t, N_EVAL_T), rec_tok, term_id, True),
             c_dom=mk(smp(ev_c, N_EVAL_C), rec_tok, term_id, False),
             c_wiki=mk(smp(wiki[len(wiki) // 2:], N_EVAL_C), rec_tok, term_id, False) if wiki else None)
    print(f"  sentences: calibration {len(D['t_train']['sents'])}+{len(D['c_train']['sents'])} | pool {len(D['t_pool']['sents'])} | eval with new words {len(D['t_eval']['sents'])}, domain control {len(D['c_dom']['sents'])}, wiki control {len(D['c_wiki']['sents']) if D['c_wiki'] else 0}")
    model.resize_token_embeddings(V0 + K)
    if model.get_output_embeddings().weight.data_ptr() != model.get_input_embeddings().weight.data_ptr(): raise RuntimeError("embeddings no longer tied after resize")
    ctx = SimpleNamespace(model=model, bos=bos, V0=V0, K=K, D=D, mu_r=mu_r, rbar=rbar)
    # directions: every variant is built from the same anchors and goes through the same calibration
    dirs, times, raw_rows = {}, {}, {}
    W_c, s_c, lam_c = fit_ridge(Dc, Rc); pred_c = (Dterm_c / s_c) @ W_c                  # centred ridge output (reference)
    dirs["ridge[all]"] = unit(pred_c)
    W_f, s_f, lam_f = fit_ridge(Dc[f_idx], Rc[f_idx]); dirs["ridge[freq5000]"] = unit((Dterm_c / s_f) @ W_f)
    # A: no centring (raw rows, no intercept)
    W_r, s_r, lam_r = fit_ridge(D_a, R_a); raw_pred = (D_in[d_term] / s_r) @ W_r
    dirs["A no-centring (cal)"] = unit(raw_pred - mu_r)
    raw_rows["A no-centring (raw output)"] = raw_pred
    # B: magnitude
    raw_rows["B own magnitude (no unit, no cal)"] = mu_r + pred_c
    raw_rows["B fixed scale 1.0 (no cal)"] = mu_r + 1.0 * rbar * unit(pred_c)
    # C: lambda
    def ridge_lam_dir(lam):
        s = float(np.linalg.norm(Dc, axis=1).mean()); X = (Dc / s).astype(np.float64); A = X.T @ X; A[np.diag_indices_from(A)] += lam * np.trace(A) / A.shape[0]
        return unit((Dterm_c / s) @ np.linalg.solve(A, X.T @ Rc.astype(np.float64)))
    LAMS = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)
    for lam in LAMS: dirs[f"C lambda={lam:g} (cal)"] = ridge_lam_dir(lam)
    mean_raw = np.stack([E[p].mean(0) for p in piece_ids]); dirs["mean-subword (cal)"] = unit(mean_raw - mu_r)
    print(f"  ridge lambda chosen by held-out anchor cosine: {lam_c} (all anchors), {lam_f} (freq5000), {lam_r} (no centring)")
    rows, scales, objs = {}, {}, {}
    for cond, d in dirs.items():
        c, o = calibrate(ctx, d); rows[cond] = mu_r + c * rbar * d; scales[cond] = c; objs[cond] = o
        print(f"  {cond}: scale {c}{'  (EDGE)' if c in (GRID[0], GRID[-1]) else ''} | train objective {o:.4f}")
    rows.update(raw_rows)
    lam_keys = [f"C lambda={lam:g} (cal)" for lam in LAMS]; lam_nll = min(lam_keys, key=lambda k: objs[k])
    print(f"  lambda chosen by the training NLL objective: {lam_nll}  (by cosine: {lam_c})")
    sel = "ridge[freq5000]" if objs["ridge[freq5000]"] < objs["ridge[all]"] else "ridge[all]"
    print(f"  ridge variant chosen by the training objective: {sel}")
    CONDS = ["original", "ridge[all]", "ridge[freq5000]", "A no-centring (cal)", "A no-centring (raw output)", "B own magnitude (no unit, no cal)", "B fixed scale 1.0 (no cal)"] + lam_keys + ["mean-subword (cal)"]
    curves = {}
    for cond in CONDS:
        t1 = time.time(); curves[cond] = {}
        if cond != "original": set_rows(ctx, rows[cond])
        curves[cond][0] = evaluate(ctx, "orig" if cond == "original" else "new")
        if TRAIN_STEPS > 0 and cond != "original":
            train_new_rows(ctx, rows[cond], TRAIN_STEPS, seed); curves[cond][TRAIN_STEPS] = evaluate(ctx, "new")
        a = curves[cond][0]["t_eval"]["tot"].mean()
        print(f"    {cond:<20} term-sentence NLL {a:.2f}" + (f" -> {curves[cond][TRAIN_STEPS]['t_eval']['tot'].mean():.2f} after {TRAIN_STEPS} steps" if TRAIN_STEPS > 0 and cond != "original" else "") + f"   [{time.time() - t1:.0f}s]")
    dmax = float((model.get_input_embeddings().weight[:V0].detach().float().cpu() - torch.as_tensor(E)).abs().max())
    print(f"  max change in the original embedding rows: {dmax:.3g}")
    assert dmax == 0.0, f"old embedding rows changed by {dmax:.3g}"
    res = dict(pair=name, domain=DOMAIN, seed=seed, K=K, terms=[g for _, g, _d in terms], dims=(E.shape[1], D_in.shape[1]), n_anchors=len(keys), curves=curves, scales=scales, objs=objs, abs_ids=dict(t_eval=[ev_abs.get(s, -1) for s in D['t_eval']['sents']], c_dom=[ev_abs.get(s, -1) for s in D['c_dom']['sents']]), 
               sel=sel, lam_cos=lam_c, lam_nll=lam_nll, map_s=times, minutes=(time.time() - t_all) / 60, train_steps=TRAIN_STEPS)
    print(f"  done in {res['minutes']:.1f} min")
    del ctx, model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return res

## Run (resumable). Pairs that fail are reported and skipped.

In [ ]:
for SEED in SEEDS:
    for pair in PAIRS:
        f = OUT / (f"{DOMAIN}__" + pair["name"].replace(" ", "_").replace("<-", "from").replace(" ", "") + f"__seed{SEED}.pkl")
        if f.exists(): print("already done:", pair["name"]); continue
        print(f"\n########## {pair['name']} | seed {SEED} ##########")
        try:
            res = run_pair(pair, SEED)
            if res is not None:
                with open(f, "wb") as fh: pickle.dump(res, fh)
        except Exception as e:
            print(f"!!! failed: {type(e).__name__}: {e}"); traceback.print_exc()
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()

## Report

In [ ]:
def boot(d, n_boot=2000):
    rng = np.random.default_rng(0); n = len(d); m = d[rng.integers(0, n, (n_boot, n))].mean(1)
    return float(d.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def fmt(d):
    m, lo, hi = boot(d); return f"{m:+.2f} [{lo:+.2f}, {hi:+.2f}]{'*' if lo > 0 or hi < 0 else ' '}"

for f in sorted(OUT.glob("*.pkl")):
    with open(f, "rb") as fh: r = pickle.load(fh)
    cv = r["curves"]; steps = [0] + ([r["train_steps"]] if r["train_steps"] > 0 else [])
    print(f"\n{'=' * 100}\n{r['pair']} | {r['domain']} | seed {r['seed']} | {r['K']} new words | lambda by cosine {r['lam_cos']}, by NLL objective: {r['lam_nll']}\n{'=' * 100}")
    o = cv["original"][0]["t_eval"]["tot"]; ref = "ridge[all]"
    for st in steps:
        print(f"\n{'zero-shot' if st == 0 else f'after {st} steps training only the new rows'} (nats per sentence; negative = better; * = CI excludes 0)")
        print(f"{'variant':<40}{'sentence NLL':>13}  {'diff vs unchanged receiver':<30}{'diff vs ridge[all] (positive = ablation worse)':<30}")
        for c in [c for c in cv if c != "original"]:
            x = cv[c][st]["t_eval"]["tot"]
            print(f"{c:<40}{x.mean():>13.2f}  {fmt(x - o):<30}{fmt(x - cv[ref][st]['t_eval']['tot']):<30}")
